# 12.6 — Graph Attention Networks

Attention lets a node choose which neighbors matter.

GAT starts from message passing and fixes uniform averaging by computing a softmax weight per edge. This notebook keeps attention local by masking to observed neighbors.

Save a copy to Drive to edit.

In [ ]:
import math
import random

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np

SEED = 12610
random.seed(SEED)
np.random.seed(SEED)


def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))


def softmax(x):
    values = np.asarray(x, dtype=float)
    shifted = values - np.max(values)
    exp_values = np.exp(shifted)
    return exp_values / exp_values.sum()


def leaky_relu(x, slope=0.2):
    values = np.asarray(x, dtype=float)
    return np.where(values >= 0.0, values, slope * values)


def graph_to_arrays(graph):
    nodes = list(graph.nodes())
    adjacency = nx.to_numpy_array(graph, nodelist=nodes, dtype=float)
    labels = np.array([graph.nodes[node].get("label", 0) for node in nodes], dtype=int)
    features = np.array([graph.nodes[node].get("x", [0.0, 0.0, 0.0]) for node in nodes], dtype=float)
    return nodes, adjacency, features, labels


def add_features(graph, labels, noise=0.05, seed=0):
    rng = np.random.default_rng(seed)
    degrees = np.array([degree for _, degree in graph.degree()], dtype=float)
    degree_scale = max(float(degrees.max()), 1.0)
    for index, node in enumerate(graph.nodes()):
        label = int(labels[index])
        base = np.array([1.0 - label, label, degrees[index] / degree_scale], dtype=float)
        graph.nodes[node]["label"] = label
        graph.nodes[node]["x"] = base + rng.normal(0.0, noise, size=3)
    return graph


def make_toy_graph():
    graph = nx.Graph()
    graph.add_edges_from([(0, 1), (1, 2), (2, 3), (0, 2)])
    labels = np.array([0, 0, 1, 1], dtype=int)
    return add_features(graph, labels, noise=0.0, seed=1)


def make_karate_graph():
    graph = nx.karate_club_graph()
    labels = np.array([0 if graph.nodes[node]["club"] == "Mr. Hi" else 1 for node in graph.nodes()], dtype=int)
    return add_features(graph, labels, noise=0.03, seed=2)


def make_sbm_graph(sizes, p_in, p_out, noise, seed):
    probs = [[p_in, p_out], [p_out, p_in]]
    graph = nx.stochastic_block_model(sizes, probs, seed=seed)
    labels = []
    for block, size in enumerate(sizes):
        labels.extend([block] * size)
    return add_features(graph, np.array(labels, dtype=int), noise=noise, seed=seed + 100)


def build_graph_ladder():
    return [
        {"name": "D1 toy 4-node", "graph": make_toy_graph(), "complexity": 1},
        {"name": "D2 karate club", "graph": make_karate_graph(), "complexity": 2},
        {"name": "D3 clean SBM", "graph": make_sbm_graph([18, 18], 0.34, 0.04, 0.07, 3), "complexity": 3},
        {"name": "D4 larger SBM", "graph": make_sbm_graph([35, 35], 0.20, 0.025, 0.10, 4), "complexity": 4},
        {"name": "D5 noisy overlap", "graph": make_sbm_graph([45, 45], 0.16, 0.075, 0.22, 5), "complexity": 5},
    ]


def class_train_mask(labels):
    mask = np.zeros(len(labels), dtype=bool)
    for label in np.unique(labels):
        candidates = np.where(labels == label)[0]
        count = max(1, len(candidates) // 4)
        mask[candidates[:count]] = True
    return mask


def centroid_predict(embeddings, labels, train_mask):
    classes = np.unique(labels)
    centroids = []
    for label in classes:
        rows = embeddings[(labels == label) & train_mask]
        centroids.append(rows.mean(axis=0))
    centroids = np.vstack(centroids)
    distances = ((embeddings[:, None, :] - centroids[None, :, :]) ** 2).sum(axis=2)
    return classes[np.argmin(distances, axis=1)]


def aligned_accuracy(predictions, labels):
    raw = float(np.mean(predictions == labels))
    flipped = float(np.mean(1 - predictions == labels))
    return max(raw, flipped)


def pairwise_auc(positive_scores, negative_scores):
    total = 0.0
    count = 0.0
    for positive in positive_scores:
        for negative in negative_scores:
            if positive > negative:
                total += 1.0
            elif positive == negative:
                total += 0.5
            count += 1.0
    return total / count


def sample_link_split(graph, seed=0, holdout_fraction=0.2):
    rng = np.random.default_rng(seed)
    edges = [tuple(sorted(edge)) for edge in graph.edges()]
    rng.shuffle(edges)
    holdout_count = max(1, int(len(edges) * holdout_fraction))
    positive_edges = edges[:holdout_count]
    train_graph = graph.copy()
    train_graph.remove_edges_from(positive_edges)
    non_edges = [tuple(sorted(edge)) for edge in nx.non_edges(graph)]
    rng.shuffle(non_edges)
    negative_edges = non_edges[:holdout_count]
    return train_graph, positive_edges, negative_edges


def spectral_embeddings(adjacency, dimensions=8):
    adjacency = np.asarray(adjacency, dtype=float)
    augmented = adjacency + np.eye(adjacency.shape[0])
    degrees = augmented.sum(axis=1)
    inv_sqrt = 1.0 / np.sqrt(np.maximum(degrees, 1e-12))
    normalized = inv_sqrt[:, None] * augmented * inv_sqrt[None, :]
    values, vectors = np.linalg.eigh(normalized)
    order = np.argsort(values)[::-1]
    keep = order[: min(dimensions, adjacency.shape[0])]
    return vectors[:, keep] * values[keep]


def plot_graph_panels(results, metric_name):
    fig, axes = plt.subplots(1, len(results), figsize=(15, 3))
    for axis, result in zip(axes, results):
        graph = result["graph"]
        colors = result.get("colors")
        pos = nx.spring_layout(graph, seed=SEED)
        nx.draw_networkx_nodes(graph, pos, node_color=colors, cmap="coolwarm", node_size=55, ax=axis)
        nx.draw_networkx_edges(graph, pos, alpha=result.get("edge_alpha", 0.25), width=0.8, ax=axis)
        axis.set_title(f"{result['name']}\n{metric_name}={result['metric']:.3f}")
        axis.axis("off")
    plt.tight_layout()
    plt.show()
    plt.figure(figsize=(5, 3))
    plt.plot([result["complexity"] for result in results], [result["metric"] for result in results], marker="o")
    plt.xticks([1, 2, 3, 4, 5], ["D1", "D2", "D3", "D4", "D5"])
    plt.ylabel(metric_name)
    plt.xlabel("graph ladder rung")
    plt.grid(True, alpha=0.3)
    plt.show()

def gat_attention(logits, values, mask=None):
    logits = np.asarray(logits, dtype=float)
    values = np.asarray(values, dtype=float)
    if mask is None:
        mask = np.ones_like(logits, dtype=bool)
    masked_logits = np.where(mask, logits, -1e9)
    weights = softmax(masked_logits)
    weights = np.where(mask, weights, 0.0)
    message = weights @ values
    return weights, message


def gat_layer(adjacency, features):
    n_nodes = adjacency.shape[0]
    transform = np.array([[1.0, -0.25, 0.15], [-0.15, 1.0, 0.25], [0.2, 0.15, 0.7]])
    hidden = features @ transform
    out = np.zeros_like(hidden)
    edge_strength = np.zeros_like(adjacency)
    attn_vec = np.array([0.7, -0.4, 0.25, 0.3, 0.2, -0.1])
    for i in range(n_nodes):
        neighbors = np.where(adjacency[i] > 0.0)[0]
        neighbors = np.unique(np.append(neighbors, i))
        logits = []
        for j in neighbors:
            pair = np.concatenate([hidden[i], hidden[j]])
            logits.append(float(leaky_relu(pair @ attn_vec)))
        weights = softmax(np.array(logits))
        out[i] = weights @ hidden[neighbors]
        edge_strength[i, neighbors] = weights
    return np.tanh(out), edge_strength


def gat_node_predictions(graph):
    nodes, adjacency, features, labels = graph_to_arrays(graph)
    embeddings, attention = gat_layer(adjacency, features)
    train_mask = class_train_mask(labels)
    predictions = centroid_predict(embeddings, labels, train_mask)
    return predictions, embeddings, attention, labels

## The concept, built once: masked attention

For neighbors $j\in N(i)$, GAT computes $\alpha_{ij}=\operatorname{softmax}_j(e_{ij})$ and $h'_i=\sum_j\alpha_{ij}Wh_j$. On the lesson query the logits are $[1,0,1]$, so the local softmax should be $[0.422,0.155,0.422]$.

In [ ]:
logits = np.array([1.0, 0.0, 1.0])
values = np.array([[1.0, 1.0], [1.0, 0.0], [2.0, 0.73575888]])
weights, message = gat_attention(logits, values)
print("weights", np.round(weights, 3))
print("message", np.round(message, 3))
assert np.allclose(np.round(logits, 3), [1.000, 0.000, 1.000])
assert np.allclose(np.round(weights, 3), [0.422, 0.155, 0.422])
assert np.allclose(np.round(message, 3), [1.267, 0.733])

A second head gives another route through the same neighborhood. Averaging heads stabilizes the message: $([1.267,0.733]+[0.8,1.2])/2=[1.033,0.967]$ after rounding.

In [ ]:
second_head = np.array([0.8, 1.2])
head_average = 0.5 * (message + second_head)
print("head average", np.round(head_average, 3))
assert np.allclose(np.round(head_average, 3), [1.033, 0.967])

## The dataset ladder

Family F11 uses one inline graph ladder for every topic: D1 is a 4-node toy graph, D2 is karate club, D3 is a stochastic block model, D4 is larger, and D5 is noisy/overlapping. The metric for this notebook is node accuracy.

In [ ]:
ladder = build_graph_ladder()
for rung in ladder:
    graph = rung["graph"]
    nodes, adjacency, features, labels = graph_to_arrays(graph)
    counts = np.bincount(labels)
    print(rung["name"], "nodes", graph.number_of_nodes(), "edges", graph.number_of_edges(), "features", features.shape, "classes", counts.tolist())
    print("sample labels", labels[:8].tolist())

## Run the same GAT layer across D1-D5

In [ ]:
results = []
for rung in ladder:
    graph = rung["graph"]
    predictions, embeddings, attention, labels = gat_node_predictions(graph)
    metric = aligned_accuracy(predictions, labels)
    results.append({"name": rung["name"], "complexity": rung["complexity"], "graph": graph, "metric": metric, "colors": predictions, "attention": attention})
    print(f"{rung['name']:<18} node-acc={metric:.3f}")

## Results visualization

In [ ]:
plot_graph_panels(results, "node-acc")

## Pitfall on D5: forgetting the mask

Softmax must run over $N(i)$. A dense softmax lets non-edges vote, turning GAT into graph-transformer-style global attention without the structural bias.

In [ ]:
d5 = ladder[-1]["graph"]
nodes, adjacency, features, labels = graph_to_arrays(d5)
masked_predictions, masked_embeddings, masked_attention, labels = gat_node_predictions(d5)
dense_adjacency = np.ones_like(adjacency) - np.eye(adjacency.shape[0])
dense_graph = nx.from_numpy_array(dense_adjacency)
for index, node in enumerate(dense_graph.nodes()):
    dense_graph.nodes[node]["label"] = int(labels[index])
    dense_graph.nodes[node]["x"] = features[index]
dense_predictions, dense_embeddings, dense_attention, dense_labels = gat_node_predictions(dense_graph)
masked_acc = aligned_accuracy(masked_predictions, labels)
dense_acc = aligned_accuracy(dense_predictions, labels)
print("wrong dense-softmax node-acc", round(dense_acc, 3))
print("fixed masked-neighbor node-acc", round(masked_acc, 3))

## Evaluate it + Practice

- Metric: node accuracy; compare it with a no-skill baseline such as majority class or random pair ranking.
- Sanity check: D1 should match the lesson arithmetic exactly before you trust D2-D5.
- Ablation: remove the attention mask or replace learned weights with uniform neighbor means
- Failure signals: non-neighbor mass is nonzero, attention saturates, or D5 collapses toward the majority class
- Baseline: predict the majority community for every node

Practice prompts:
1. Change one graph-ladder noise value and predict which rung should move most.

2. Add a second diagnostic printout that exposes one intermediate tensor for D3.

3. Replace the simple readout with another CPU-only NumPy readout and compare the metric curve.